In [0]:
dbutils.widgets.removeAll()

In [0]:
ProcessedJSON = {
    "FileIds": [
        {
            "ClientID": "01",
            "FileID": "01",
            "FileName": "member.csv",
            "ClientContainer": "/Workspace/Users/logi@openhealthagents.org/edi/processed",
            "CurrentFolderPath": "",
            "ProcessedFolderPath": "/Volumes/edi/bronze/preprocess",
            "ColumnDelimiter": ",",
            "HasHeader": "true",
            "IgnoreHeader": "False",
            "FileLayoutID": "834",
            "FileLayoutDescription": "Standard834",
            "SchemaFileName": "member_7.12_schema.json",
            "SchemaFilePath": "/Workspace/Users/logi@openhealthagents.org/edi/datalake-dev/JSON/Schema",
            "TextQualifier": "\""
        }
    ]
}

In [0]:
# Create widget for ProcessedJSON parameter
dbutils.widgets.text("ProcessedJSON", "", "")
widget_value = dbutils.widgets.get("ProcessedJSON")
if widget_value:  # Only override if widget has a value
    ProcessedJSON = widget_value

In [0]:
# Define notebook paths
fcfNotebook = "../DatalakeProcessing/FCFClaimsProcessing"
procNotebook = "../DatalakeProcessing/MoveFileToProcess"

In [0]:
from pyspark.sql.functions import explode, col
import json

In [0]:
%run "/Workspace/Users/logi@openhealthagents.org/edi/databricks-dev/Notebooks/DatalakeProcessing/MoveFileToProcess"

In [0]:
%run "/Workspace/Users/logi@openhealthagents.org/edi/databricks-dev/CommonMethods/ABC/SynJSONCreatorClass"

In [0]:
%run "/Workspace/Users/logi@openhealthagents.org/edi/databricks-dev/CommonMethods/ABC/FileHandling"

In [0]:
# Handle both dict (from Cell 1) and string (from widget) formats
if isinstance(ProcessedJSON, str):
    if not ProcessedJSON or ProcessedJSON.strip() == "":
        raise ValueError("ProcessedJSON parameter is required")
    ProcessedJSON = json.loads(ProcessedJSON)
elif not ProcessedJSON:
    raise ValueError("ProcessedJSON parameter is required")

filesDF = spark.createDataFrame([ProcessedJSON])

explodedFileIDs = filesDF.select(explode(col("FileIds"))).select(
    col("col.ClientID")
    ,col("col.FileID")
    ,col("col.FileName")
    ,col("col.ClientContainer")
    ,col("col.CurrentFolderPath")
    ,col("col.ProcessedFolderPath")
    ,col("col.ColumnDelimiter")
    ,col("col.HasHeader")
    ,col("col.IgnoreHeader")
    ,col("col.FileLayoutID")
    ,col("col.FileLayoutDescription")
    ,col("col.SchemaFileName")
    ,col("col.SchemaFilePath")
    ,col("col.TextQualifier")
)

ErrorMessage = ""
doubleQuote = '"'

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

try:
    currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
except Exception:
    currentJobId = "Undefined"

rJSON = synJSONCreator()

rJSON.addBracketStart()

iterator = 0
exploded_pd = explodedFileIDs.toPandas()

if exploded_pd.empty:
    raise ValueError("No files found in explodedFileIDs. Please check the input JSON or upstream processing.")

for index, t in exploded_pd.iterrows():
    print(f"Begin {t['FileID']}-{t['FileName']}")
    CurrPath = f"{t['ClientContainer']}{t['CurrentFolderPath']}"
    # FIX: Use ProcessedFolderPath as absolute, do not prepend ClientContainer
    ProcessedPath = t['ProcessedFolderPath']
    SchemaFile = f"{t['SchemaFilePath']}/{t['SchemaFileName']}"
    FullFileName = f"{t['ClientContainer']}{t['CurrentFolderPath']}/{t['FileName']}"

    print(f"--> Target Data Path: file:{FullFileName}")
    print(f"--> Target Schema Path: file:{SchemaFile}")
    print(f"--> Target Output Path: {ProcessedPath}")

    try:
        dbutils.fs.ls(f"file:{FullFileName}")
        f = True
    except Exception as e:
        print(f"Data file check failed with error: {str(e)}")
        f = False

    try:
        dbutils.fs.ls(f"file:{SchemaFile}")
        s = True
    except Exception as e:
        print(f"Schema file check failed with error: {str(e)}")
        s = False

    if f == True and s == True:
        if iterator != 0:
            rJSON.addComma()

        rJSON.addBraceStart()
        rJSON.addNewEntry("FileID", t['FileID'])
        rJSON.addNewEntry("FileName", t['FileName'])
        try:
            if t['FileLayoutDescription'] == "FCF":
                results = process_fcf_claims(
                    ClientId=t['ClientID'],
                    FileId=t['FileID'],
                    FileLayoutId=t['FileLayoutID'],
                    FileLayoutDescription=t['FileLayoutDescription'],
                    ColumnDelimiter=t['ColumnDelimiter'],
                    HasHeader=t['HasHeader'],
                    IgnoreHeader=t['IgnoreHeader'],
                    textQualifier=t['TextQualifier'],
                    FullFileName=FullFileName,
                    SchemaFile=SchemaFile,
                    ProcessedPath=ProcessedPath
                )
            else:
                results = process_move_file(
                    ClientId=t['ClientID'],
                    FileId=t['FileID'],
                    FileLayoutId=t['FileLayoutID'],
                    FileLayoutDescription=t['FileLayoutDescription'],
                    ColumnDelimiter=t['ColumnDelimiter'],
                    HasHeader=t['HasHeader'],
                    IgnoreHeader=t['IgnoreHeader'],
                    textQualifier=t['TextQualifier'],
                    FullFileName=FullFileName,
                    SchemaFile=SchemaFile,
                    ProcessedPath=ProcessedPath
                )

            returnedJson = spark.createDataFrame([json.loads(str(results))])

            for x in returnedJson.collect():
                rJSON.addNewEntry("FullFilePath", ProcessedPath)
                rJSON.addNewEntry("CurrentJobId", x['CurrentJobId'])
                rJSON.addNewEntry("Status", x['Status']) 
                rJSON.addNewEntry("RecordCount", x['ProcessedCount'])     
                rJSON.addNewEntry("ErrorMessage", x['ErrorMessage'], False)                                    
                
        except Exception as e:
            clean_err = str(e).strip().replace(doubleQuote, "")
            rJSON.addNewEntry("CurrentJobId", "Undefined")
            rJSON.addNewEntry("Status", "FAILURE")
            rJSON.addNewEntry("RecordCount", "")   
            rJSON.addNewEntry("ErrorMessage", clean_err, False)   
  
        rJSON.addBraceEnd()
  
    elif f == False:
        if iterator != 0: rJSON.addComma()
        rJSON.addBraceStart()
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("FileID", t['FileID'])
        rJSON.addNewEntry("FileName", t['FileName'])
        rJSON.addNewEntry("FullFilePath", CurrPath)
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", "Data File Not Found", False)
        rJSON.addBraceEnd()
        
    elif s == False:
        if iterator != 0: rJSON.addComma()
        rJSON.addBraceStart()
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("FileID", t['FileID'])
        rJSON.addNewEntry("FileName", t['SchemaFileName'])
        rJSON.addNewEntry("FullFilePath", CurrPath)
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", "Schema File Not Found", False)
        rJSON.addBraceEnd()

    iterator += 1

rJSON.addBracketEnd()

returnVal = rJSON.getJSON()
print(returnVal)
dbutils.notebook.exit(returnVal)